# Phase 0: Ist-Aufnahme — Dateiorganisations-Agent (v2)

Scannt einen Ordner rekursiv, erkennt Probleme, zeigt einen **interaktiven Report**.

### Neuerungen v2
- **Pfade sind kopierbar** — jedes Problem zeigt den vollständigen Pfad
- **Severity-Filter** — Tabs für S1/S2/S3/S4, nur relevante Probleme sehen
- **Explorer-Öffnung** — Pfad eingeben → Ordner öffnet sich im Windows-Explorer
- **Ordner-Detail** — einzelne Ordner inspizieren mit allen Dateien + Problemen
- **Kompakter JSON-Export** — wahlweise ohne die riesige `alle_dateien`-Liste

### Was passiert?
- Rein lesend — keine Dateien werden verändert
- SHA-256-Hashes zur Duplikat-Erkennung
- Erkennung: generische Namen, Versionierung, Sonderzeichen, Tiefe, Ausreißer

## 1. Gradio installieren

In [ ]:
try:
    import gradio as gr
    print(f"Gradio {gr.__version__} ist bereits installiert.")
except ImportError:
    print("Gradio wird installiert...")
    %pip install gradio -q
    import gradio as gr
    print(f"Gradio {gr.__version__} installiert.")

## 2. Scanner-Code

In [ ]:
import os
import sys
import json
import hashlib
import re
import subprocess
import platform
import tempfile
from pathlib import Path
from datetime import datetime
from collections import Counter, defaultdict
from dataclasses import dataclass, field, asdict
from typing import Optional


# ════════════════════════════════════════════════════════════════
# Konfiguration
# ════════════════════════════════════════════════════════════════

IGNORIERTE_VERZEICHNISSE = {
    "node_modules", ".git", "__pycache__", ".ipynb_checkpoints",
    ".venv", "venv", "env", ".env", ".tox", ".mypy_cache",
    ".pytest_cache", "dist", "build", ".next", ".nuxt",
}

IGNORIERTE_DATEIEN = {
    ".DS_Store", "Thumbs.db", "desktop.ini", ".gitkeep",
}

LESBAR_VOLL = {
    ".ipynb", ".py", ".js", ".ts", ".jsx", ".tsx",
    ".html", ".css", ".scss", ".less",
    ".ino", ".cpp", ".c", ".h", ".hpp",
    ".md", ".txt", ".rst", ".log", ".ini", ".cfg", ".toml", ".yaml", ".yml",
    ".csv", ".tsv", ".json", ".xml",
    ".docx", ".pptx", ".xlsx",
    ".svg", ".sh", ".bat", ".ps1",
    ".r", ".R", ".jl", ".lua", ".rb", ".php",
    ".sql", ".tex", ".bib",
}

LESBAR_TEILWEISE = {
    ".pdf", ".zip", ".tar", ".gz", ".7z", ".rar",
}

NICHT_LESBAR = {
    ".pth", ".pkl", ".joblib", ".h5", ".hdf5", ".onnx",
    ".sps", ".hex", ".bin", ".dat",
    ".png", ".jpg", ".jpeg", ".gif", ".bmp", ".webp", ".ico", ".tiff",
    ".mp4", ".avi", ".mov", ".mkv", ".wmv", ".flv",
    ".mp3", ".wav", ".flac", ".ogg", ".aac", ".wma",
    ".exe", ".dll", ".so", ".dylib",
    ".o", ".obj", ".class", ".pyc", ".pyo",
}

DOMAENE_UNTERRICHT = {".docx", ".pdf", ".pptx", ".odt", ".odp", ".ods"}
DOMAENE_CODE = {
    ".ipynb", ".py", ".js", ".ts", ".jsx", ".tsx", ".html", ".css",
    ".ino", ".cpp", ".c", ".h", ".csv", ".json",
    ".pth", ".pkl", ".h5", ".onnx", ".joblib",
}

GENERISCHE_NAMEN = re.compile(
    r"^(Untitled\d*|Copy of |Kopie von |.*\(\d+\)$|Neues Dokument|Dokument\d*|"
    r"Unbenannt\d*|Mappe\d*|Tabelle\d*|Präsentation\d*)",
    re.IGNORECASE,
)

VERSIONIERUNGS_MUSTER = re.compile(
    r"(_final|_v\d+|_neu|_alt|_aktuell|_Kopie|_copy|_backup|_old|_new|"
    r"_FINAL|_V\d+|_NEU|_ALT)",
)

PROBLEMATISCHE_ZEICHEN = re.compile(r"[#%&!@\$\^]|  +|\.\.")

print("Konfiguration geladen.")

In [ ]:
# ════════════════════════════════════════════════════════════════
# Datenstrukturen
# ════════════════════════════════════════════════════════════════

@dataclass
class DateiInfo:
    pfad: str
    name: str
    erweiterung: str
    groesse_bytes: int
    erstellt: str
    geaendert: str
    sha256: Optional[str] = None
    lesbar: str = "unbekannt"
    ist_generisch: bool = False
    hat_versionierung: bool = False
    hat_problematische_zeichen: bool = False

    @property
    def groesse_lesbar(self) -> str:
        b = self.groesse_bytes
        for einheit in ["B", "KB", "MB", "GB"]:
            if b < 1024:
                return f"{b:.1f} {einheit}"
            b /= 1024
        return f"{b:.1f} TB"


@dataclass
class OrdnerInfo:
    pfad: str
    tiefe: int
    anzahl_dateien: int = 0
    anzahl_unterordner: int = 0
    dateitypen: dict = field(default_factory=dict)
    groesste_datei: Optional[str] = None
    aelteste_datei: Optional[str] = None
    neueste_datei: Optional[str] = None
    benennungsmuster: dict = field(default_factory=dict)
    hash_duplikate: list = field(default_factory=list)
    vermutete_domaene: str = "unbekannt"
    dateien: list = field(default_factory=list)
    ist_leer: bool = False
    ist_einsam: bool = False
    ist_rekursiv_leer: bool = False


@dataclass
class ScanErgebnis:
    eingabepfad: str
    scan_zeitpunkt: str
    gesamtdateien: int = 0
    gesamtordner: int = 0
    gesamtgroesse_bytes: int = 0
    ordner: list = field(default_factory=list)
    alle_dateien: list = field(default_factory=list)
    hash_duplikate_global: list = field(default_factory=list)
    dateityp_verteilung_global: dict = field(default_factory=dict)
    tiefe_max: int = 0
    probleme_phase0: list = field(default_factory=list)

print("Datenstrukturen definiert.")

In [ ]:
# ════════════════════════════════════════════════════════════════
# Scanner-Klasse (identisch mit v1)
# ════════════════════════════════════════════════════════════════

class Phase0Scanner:

    def __init__(self, eingabepfad: str, max_hash_size_mb: int = 200):
        self.eingabepfad = Path(eingabepfad).resolve()
        self.max_hash_size = max_hash_size_mb * 1024 * 1024
        self.alle_dateien: list[DateiInfo] = []
        self.alle_ordner: list[OrdnerInfo] = []
        self.hash_map: dict[str, list[str]] = defaultdict(list)

    # ─── Hilfsfunktionen ────────────────────────────────────

    @staticmethod
    def _zeitstempel(ts: float) -> str:
        try:
            return datetime.fromtimestamp(ts).isoformat(timespec="seconds")
        except (OSError, ValueError):
            return "unbekannt"

    def _sha256(self, pfad: Path) -> Optional[str]:
        try:
            if pfad.stat().st_size > self.max_hash_size:
                return None
            h = hashlib.sha256()
            with open(pfad, "rb") as f:
                while chunk := f.read(8192):
                    h.update(chunk)
            return h.hexdigest()
        except (PermissionError, OSError):
            return None

    @staticmethod
    def _lesbarkeit(erweiterung: str) -> str:
        ext = erweiterung.lower()
        if ext in LESBAR_VOLL:
            return "voll"
        if ext in LESBAR_TEILWEISE:
            return "teilweise"
        if ext in NICHT_LESBAR:
            return "nein"
        return "unbekannt"

    @staticmethod
    def _domaene_aus_verteilung(dateitypen: dict) -> str:
        gesamt = sum(dateitypen.values())
        if gesamt == 0:
            return "leer"
        unterricht = sum(dateitypen.get(ext, 0) for ext in DOMAENE_UNTERRICHT)
        code = sum(dateitypen.get(ext, 0) for ext in DOMAENE_CODE)
        anteil_unterricht = unterricht / gesamt
        anteil_code = code / gesamt
        if anteil_unterricht >= 0.70:
            return "unterricht"
        if anteil_code >= 0.70:
            return "code"
        if anteil_unterricht > 0 and anteil_code > 0:
            return "gemischt"
        return "unbestimmt"

    def _ist_ignoriert(self, name: str, ist_ordner: bool) -> bool:
        if ist_ordner:
            return name in IGNORIERTE_VERZEICHNISSE or name.startswith(".")
        return name in IGNORIERTE_DATEIEN

    # ─── Benennungsmuster ──────────────────────────────────

    @staticmethod
    def _muster_erkennen(dateinamen: list[str]) -> dict:
        if not dateinamen:
            return {"erkannt": None, "konfidenz": 0.0, "abweichler": []}

        trennzeichen = Counter()
        for name in dateinamen:
            stamm = Path(name).stem
            if "_" in stamm:
                trennzeichen["_"] += 1
            if "-" in stamm:
                trennzeichen["-"] += 1
            if " " in stamm:
                trennzeichen[" "] += 1
            if not any(c in stamm for c in "_ -"):
                trennzeichen["keins"] += 1

        praefixe = Counter()
        dominant_sep = trennzeichen.most_common(1)[0][0] if trennzeichen else "_"
        if dominant_sep == "keins":
            dominant_sep = "_"

        for name in dateinamen:
            stamm = Path(name).stem
            teile = stamm.split(dominant_sep) if dominant_sep != "keins" else [stamm]
            if teile:
                praefixe[teile[0]] += 1

        haeufigste_praefixe = [p for p, c in praefixe.most_common(10) if c >= 2]

        muster_treffer = 0
        abweichler = []

        if haeufigste_praefixe:
            schema = "{Präfix}" + dominant_sep + "{Thema}"
            for name in dateinamen:
                stamm = Path(name).stem
                teile = stamm.split(dominant_sep, 1)
                if teile[0] in haeufigste_praefixe and len(teile) > 1:
                    muster_treffer += 1
                else:
                    abweichler.append(name)
        else:
            schema = None
            abweichler = dateinamen[:]

        konfidenz = muster_treffer / len(dateinamen) if dateinamen else 0.0

        return {
            "erkannt": schema,
            "trennzeichen": dominant_sep,
            "haeufige_praefixe": haeufigste_praefixe,
            "konfidenz": round(konfidenz, 2),
            "abweichler": abweichler,
        }

    # ─── Einzelne Datei scannen ────────────────────────────

    def _datei_scannen(self, pfad: Path) -> DateiInfo:
        stat = pfad.stat()
        erweiterung = pfad.suffix.lower()
        sha = self._sha256(pfad)
        if sha:
            self.hash_map[sha].append(str(pfad))
        name_ohne_ext = pfad.stem
        return DateiInfo(
            pfad=str(pfad),
            name=pfad.name,
            erweiterung=erweiterung,
            groesse_bytes=stat.st_size,
            erstellt=self._zeitstempel(stat.st_ctime),
            geaendert=self._zeitstempel(stat.st_mtime),
            sha256=sha,
            lesbar=self._lesbarkeit(erweiterung),
            ist_generisch=bool(GENERISCHE_NAMEN.match(name_ohne_ext)),
            hat_versionierung=bool(VERSIONIERUNGS_MUSTER.search(name_ohne_ext)),
            hat_problematische_zeichen=bool(PROBLEMATISCHE_ZEICHEN.search(pfad.name)),
        )

    # ─── Einzelnen Ordner scannen ──────────────────────────

    def _ordner_scannen(self, pfad: Path, tiefe: int) -> OrdnerInfo:
        dateien_im_ordner: list[DateiInfo] = []
        unterordner_namen: list[str] = []

        try:
            eintraege = list(pfad.iterdir())
        except PermissionError:
            return OrdnerInfo(pfad=str(pfad), tiefe=tiefe, ist_leer=True, ist_rekursiv_leer=True)

        for eintrag in sorted(eintraege, key=lambda e: e.name.lower()):
            if eintrag.is_dir():
                if not self._ist_ignoriert(eintrag.name, ist_ordner=True):
                    unterordner_namen.append(eintrag.name)
            elif eintrag.is_file():
                if not self._ist_ignoriert(eintrag.name, ist_ordner=False):
                    try:
                        di = self._datei_scannen(eintrag)
                        dateien_im_ordner.append(di)
                        self.alle_dateien.append(di)
                    except (PermissionError, OSError):
                        pass

        typ_counter = Counter(d.erweiterung for d in dateien_im_ordner)
        dateitypen = dict(typ_counter.most_common())
        groesste = max(dateien_im_ordner, key=lambda d: d.groesse_bytes, default=None)
        aelteste = min(dateien_im_ordner, key=lambda d: d.geaendert, default=None)
        neueste = max(dateien_im_ordner, key=lambda d: d.geaendert, default=None)
        muster = self._muster_erkennen([d.name for d in dateien_im_ordner])
        domaene = self._domaene_aus_verteilung(dateitypen)

        pfade_im_ordner = {d.pfad for d in dateien_im_ordner}
        duplikate_lokal = []
        for sha, pfade in self.hash_map.items():
            lokale = [p for p in pfade if p in pfade_im_ordner]
            if len(lokale) > 1:
                duplikate_lokal.append(lokale)

        return OrdnerInfo(
            pfad=str(pfad), tiefe=tiefe,
            anzahl_dateien=len(dateien_im_ordner),
            anzahl_unterordner=len(unterordner_namen),
            dateitypen=dateitypen,
            groesste_datei=f"{groesste.name} ({groesste.groesse_lesbar})" if groesste else None,
            aelteste_datei=f"{aelteste.name} ({aelteste.geaendert})" if aelteste else None,
            neueste_datei=f"{neueste.name} ({neueste.geaendert})" if neueste else None,
            benennungsmuster=muster,
            hash_duplikate=duplikate_lokal,
            vermutete_domaene=domaene,
            dateien=[asdict(d) for d in dateien_im_ordner],
            ist_leer=(len(dateien_im_ordner) == 0 and len(unterordner_namen) == 0),
            ist_einsam=(len(dateien_im_ordner) == 1 and len(unterordner_namen) == 0),
        )

    # ─── Rekursiver Scan ───────────────────────────────────

    def _rekursiv_scannen(self, pfad: Path, tiefe: int = 0):
        ordner_info = self._ordner_scannen(pfad, tiefe)
        self.alle_ordner.append(ordner_info)
        try:
            for eintrag in sorted(pfad.iterdir(), key=lambda e: e.name.lower()):
                if eintrag.is_dir() and not self._ist_ignoriert(eintrag.name, ist_ordner=True):
                    self._rekursiv_scannen(eintrag, tiefe + 1)
        except PermissionError:
            pass

    # ─── Rekursiv-leer prüfen ──────────────────────────────

    def _rekursiv_leer_pruefen(self):
        pfad_zu_ordner = {o.pfad: o for o in self.alle_ordner}
        kinder = defaultdict(list)
        for o in self.alle_ordner:
            eltern = str(Path(o.pfad).parent)
            if eltern in pfad_zu_ordner:
                kinder[eltern].append(o.pfad)
        for ordner in sorted(self.alle_ordner, key=lambda o: o.tiefe, reverse=True):
            if ordner.anzahl_dateien > 0:
                ordner.ist_rekursiv_leer = False
                continue
            kind_pfade = kinder.get(ordner.pfad, [])
            if not kind_pfade:
                ordner.ist_rekursiv_leer = ordner.ist_leer
            else:
                ordner.ist_rekursiv_leer = all(
                    pfad_zu_ordner[kp].ist_rekursiv_leer for kp in kind_pfade
                )

    # ─── Globale Duplikate ─────────────────────────────────

    def _globale_duplikate(self) -> list:
        return [
            {"sha256": sha, "dateien": pfade, "anzahl": len(pfade)}
            for sha, pfade in self.hash_map.items()
            if len(pfade) > 1
        ]

    # ─── Probleme sammeln ──────────────────────────────────

    def _probleme_sammeln(self) -> list:
        probleme = []

        for ordner in self.alle_ordner:
            pfad = ordner.pfad

            if ordner.tiefe > 6:
                probleme.append({"regel": "R-U02", "severity": "S2", "pfad": pfad,
                    "beschreibung": f"Tiefe {ordner.tiefe} — dringend vereinfachen"})
            elif ordner.tiefe > 4:
                probleme.append({"regel": "R-U02", "severity": "S3", "pfad": pfad,
                    "beschreibung": f"Tiefe {ordner.tiefe} — Vereinfachung prüfen"})

            if ordner.ist_einsam:
                probleme.append({"regel": "R-U03", "severity": "S3", "pfad": pfad,
                    "beschreibung": "Ordner enthält genau 1 Datei"})

            if ordner.ist_rekursiv_leer:
                probleme.append({"regel": "R-U04", "severity": "S4", "pfad": pfad,
                    "beschreibung": "Ordner ist rekursiv leer"})

            for dup_gruppe in ordner.hash_duplikate:
                probleme.append({"regel": "R-U05", "severity": "S1", "pfad": pfad,
                    "beschreibung": f"Hash-Duplikate: {dup_gruppe}"})

        for datei in self.alle_dateien:
            if datei.hat_problematische_zeichen:
                probleme.append({"regel": "R-U08", "severity": "S4", "pfad": datei.pfad,
                    "beschreibung": f"Problematische Zeichen in: {datei.name}"})
            if datei.hat_versionierung:
                probleme.append({"regel": "R-U09", "severity": "S2", "pfad": datei.pfad,
                    "beschreibung": f"Implizite Versionierung: {datei.name}"})
            if datei.ist_generisch:
                probleme.append({"regel": "R-C01", "severity": "S1", "pfad": datei.pfad,
                    "beschreibung": f"Generischer Dateiname: {datei.name}"})

        # R-U07: Dateityp-Ausreißer (domänenbewusst)
        begleiter_unterricht = {
            ".docx", ".pdf", ".pptx", ".odt", ".odp", ".ods", ".xlsx",
            ".png", ".jpg", ".jpeg", ".gif", ".svg", ".mp4", ".wav",
        }
        begleiter_code = {
            ".ipynb", ".py", ".js", ".ts", ".jsx", ".tsx",
            ".html", ".css", ".json", ".csv", ".tsv", ".xlsx",
            ".pth", ".pkl", ".h5", ".onnx", ".joblib",
            ".txt", ".md", ".yaml", ".yml", ".toml", ".cfg", ".ini",
            ".sh", ".bat", ".pyc", ".pyo",
        }
        for ordner in self.alle_ordner:
            gesamt = ordner.anzahl_dateien
            if gesamt < 5:
                continue
            domaene = ordner.vermutete_domaene
            if domaene == "unterricht":
                erlaubt = begleiter_unterricht
            elif domaene == "code":
                erlaubt = begleiter_code
            else:
                erlaubt = begleiter_unterricht | begleiter_code
            for ext, count in ordner.dateitypen.items():
                anteil = count / gesamt
                if anteil < 0.20 and count <= 2 and ext not in erlaubt:
                    probleme.append({"regel": "R-U07", "severity": "S2", "pfad": ordner.pfad,
                        "beschreibung": f"Dateityp-Ausreißer: {count}× {ext} unter {gesamt} Dateien ({anteil:.0%})"})

        # R-C04: Artefakt-Dateien
        for datei in self.alle_dateien:
            if datei.erweiterung in {".pyc", ".pyo"}:
                probleme.append({"regel": "R-C04", "severity": "S4", "pfad": datei.pfad,
                    "beschreibung": f"Artefakt-Datei: {datei.name}"})

        return probleme

    # ─── Hauptmethode ──────────────────────────────────────

    def scannen(self) -> dict:
        """Führt den vollständigen Scan durch. Gibt dict zurück."""
        if not self.eingabepfad.exists():
            raise FileNotFoundError(f"Pfad existiert nicht: {self.eingabepfad}")
        if not self.eingabepfad.is_dir():
            raise NotADirectoryError(f"Pfad ist kein Verzeichnis: {self.eingabepfad}")

        self._rekursiv_scannen(self.eingabepfad)
        self._rekursiv_leer_pruefen()
        duplikate = self._globale_duplikate()
        probleme = self._probleme_sammeln()
        typ_counter = Counter(d.erweiterung for d in self.alle_dateien)
        max_tiefe = max((o.tiefe for o in self.alle_ordner), default=0)
        gesamtgroesse = sum(d.groesse_bytes for d in self.alle_dateien)

        ergebnis = ScanErgebnis(
            eingabepfad=str(self.eingabepfad),
            scan_zeitpunkt=datetime.now().isoformat(timespec="seconds"),
            gesamtdateien=len(self.alle_dateien),
            gesamtordner=len(self.alle_ordner),
            gesamtgroesse_bytes=gesamtgroesse,
            ordner=[asdict(o) for o in self.alle_ordner],
            alle_dateien=[asdict(d) for d in self.alle_dateien],
            hash_duplikate_global=duplikate,
            dateityp_verteilung_global=dict(typ_counter.most_common()),
            tiefe_max=max_tiefe,
            probleme_phase0=probleme,
        )
        return asdict(ergebnis)

print("Scanner-Klasse geladen.")

## 3. Bericht-Formatierung (v2 — mit kopierbaren Pfaden)

In [ ]:
def bericht_uebersicht(e: dict) -> str:
    """Kompakte Übersicht: Statistik + Dateitypen + Domänen."""
    zeilen = []
    z = zeilen.append

    z("## Scan-Ergebnis")
    z("")
    z(f"| Eigenschaft | Wert |")
    z(f"|:---|:---|")
    z(f"| **Pfad** | `{e['eingabepfad']}` |")
    z(f"| **Zeitpunkt** | {e['scan_zeitpunkt']} |")
    z(f"| **Dateien** | {e['gesamtdateien']:,} |")
    z(f"| **Ordner** | {e['gesamtordner']:,} |")
    z(f"| **Gesamtgröße** | {e['gesamtgroesse_bytes'] / (1024**2):.1f} MB |")
    z(f"| **Max. Tiefe** | {e['tiefe_max']} |")
    z("")

    # Dateityp-Verteilung (Top 20)
    z("### Dateityp-Verteilung")
    z("")
    z("| Typ | Anzahl | |")
    z("|:---|---:|:---|")
    typen = sorted(e['dateityp_verteilung_global'].items(), key=lambda x: -x[1])
    for ext, count in typen[:20]:
        balken = '█' * min(count, 30)
        z(f"| `{ext}` | {count} | {balken} |")
    if len(typen) > 20:
        rest = sum(c for _, c in typen[20:])
        z(f"| *... {len(typen)-20} weitere* | {rest} | |")
    z("")

    # Domänen
    domaenen = Counter(
        o["vermutete_domaene"] for o in e["ordner"] if o["anzahl_dateien"] > 0
    )
    z("### Domänen-Verteilung")
    z("")
    z("| Domäne | Ordner |")
    z("|:---|---:|")
    for dom, count in domaenen.most_common():
        z(f"| {dom} | {count} |")
    z("")

    # Severity-Zusammenfassung
    probs = e['probleme_phase0']
    sev_count = Counter(p['severity'] for p in probs)
    z("### Probleme gesamt")
    z("")
    z(f"| Severity | Anzahl |")
    z(f"|:---|---:|")
    z(f"| 🔴 S1 (Kritisch) | {sev_count.get('S1', 0)} |")
    z(f"| 🟠 S2 (Hoch) | {sev_count.get('S2', 0)} |")
    z(f"| 🟡 S3 (Mittel) | {sev_count.get('S3', 0)} |")
    z(f"| 🔵 S4 (Niedrig) | {sev_count.get('S4', 0)} |")
    z(f"| **Gesamt** | **{len(probs)}** |")
    z("")

    return "\n".join(zeilen)


def _problem_zeile(p: dict) -> str:
    """Eine einzelne Problem-Zeile mit kopierbardem Pfad."""
    pfad = p.get('pfad', '?')
    # Ordner-Pfad: alles bis zum letzten Separator
    ordner = str(Path(pfad).parent) if not Path(pfad).is_dir() else pfad
    return (
        f"**[{p['regel']}]** {p['beschreibung']}\n"
        f"```\n{pfad}\n```\n"
    )


def bericht_severity(e: dict, severity: str) -> str:
    """Alle Probleme einer bestimmten Severity, mit kopierbaren Pfaden."""
    icons = {"S1": "🔴", "S2": "🟠", "S3": "🟡", "S4": "🔵"}
    labels = {"S1": "Kritisch", "S2": "Hoch", "S3": "Mittel", "S4": "Niedrig"}
    probs = [p for p in e['probleme_phase0'] if p['severity'] == severity]

    if not probs:
        return f"### {icons.get(severity, '')} {severity}: Keine Probleme ✓"

    zeilen = [f"### {icons.get(severity, '')} {severity} ({labels.get(severity, '')}): {len(probs)} Probleme\n"]

    # Gruppiert nach Regel
    nach_regel = defaultdict(list)
    for p in probs:
        nach_regel[p['regel']].append(p)

    for regel, gruppe in sorted(nach_regel.items()):
        zeilen.append(f"#### {regel} — {len(gruppe)} Treffer\n")
        # Zeige max. 50 pro Regel, danach Hinweis
        for p in gruppe[:50]:
            zeilen.append(_problem_zeile(p))
        if len(gruppe) > 50:
            zeilen.append(f"*... und {len(gruppe) - 50} weitere Treffer für {regel}*\n")

    return "\n".join(zeilen)


def bericht_duplikate(e: dict) -> str:
    """Hash-Duplikate mit kopierbaren Pfaden."""
    dups = e['hash_duplikate_global']
    if not dups:
        return "### Hash-Duplikate: Keine gefunden ✓"

    zeilen = [f"### Hash-Duplikate: {len(dups)} Gruppen\n"]
    zeilen.append("Jede Gruppe enthält identische Dateien (gleicher SHA-256).\n")
    zeilen.append("**Tipp:** Pro Gruppe nur eine Datei behalten, Rest löschen.\n")

    for i, dup in enumerate(dups[:30], 1):
        zeilen.append(f"**Gruppe {i}** — {dup['anzahl']} identische Dateien:")
        for p in dup['dateien']:
            zeilen.append(f"```\n{p}\n```")
        zeilen.append("")

    if len(dups) > 30:
        zeilen.append(f"*... und {len(dups) - 30} weitere Gruppen*")

    return "\n".join(zeilen)


print("Bericht-Formatierung v2 geladen.")

## 4. Ordner-Detail und Explorer-Öffnung

In [ ]:
def ordner_vorschau(pfad: str) -> str:
    """Zeigt die ersten 2 Ebenen eines Ordners zur Orientierung."""
    if not pfad or not pfad.strip():
        return "*Bitte einen Pfad eingeben.*"

    p = Path(pfad.strip())
    if not p.exists():
        return f"⚠️ Pfad existiert nicht: `{p}`"
    if not p.is_dir():
        return f"⚠️ Pfad ist kein Ordner: `{p}`"

    zeilen = [f"📁 **{p.name}/**\n"]
    try:
        eintraege = sorted(p.iterdir(), key=lambda e: (not e.is_dir(), e.name.lower()))
    except PermissionError:
        return f"⚠️ Zugriff verweigert: `{p}`"

    for i, eintrag in enumerate(eintraege):
        if i >= 30:
            zeilen.append(f"  *... und {len(eintraege) - 30} weitere Einträge*")
            break
        if eintrag.is_dir():
            zeilen.append(f"  📁 {eintrag.name}/")
            try:
                sub = sorted(eintrag.iterdir(), key=lambda e: (not e.is_dir(), e.name.lower()))
                for j, sub_eintrag in enumerate(sub[:8]):
                    icon = "📁" if sub_eintrag.is_dir() else "📄"
                    zeilen.append(f"    {icon} {sub_eintrag.name}")
                if len(sub) > 8:
                    zeilen.append(f"    *... +{len(sub) - 8} weitere*")
            except PermissionError:
                zeilen.append(f"    ⚠️ Zugriff verweigert")
        else:
            zeilen.append(f"  📄 {eintrag.name}")

    return "\n".join(zeilen)


def ordner_detail(e: dict, pfad_suche: str) -> str:
    """Zeigt alle Infos und Probleme für einen bestimmten Ordner."""
    if not e or not pfad_suche or not pfad_suche.strip():
        return "*Zuerst Scan durchführen, dann Ordnerpfad eingeben.*"

    pfad_suche = pfad_suche.strip().rstrip('/').rstrip('\\\\')

    # Ordner finden (exakt oder Teilstring)
    treffer = []
    for o in e.get('ordner', []):
        op = o['pfad'].rstrip('/').rstrip('\\\\')
        if op == pfad_suche or op.replace('/', '\\\\') == pfad_suche.replace('/', '\\\\'):
            treffer = [o]
            break
        if pfad_suche.lower() in op.lower():
            treffer.append(o)

    if not treffer:
        return f"Kein Ordner gefunden für: `{pfad_suche}`\n\nTipp: Pfad aus dem Problembericht kopieren."

    if len(treffer) > 1:
        zeilen = [f"Mehrere Treffer für `{pfad_suche}` — bitte präzisieren:\n"]
        for o in treffer[:20]:
            zeilen.append(f"```\n{o['pfad']}\n```")
        return "\n".join(zeilen)

    o = treffer[0]
    zeilen = []
    z = zeilen.append

    z(f"## 📁 Ordner-Detail")
    z(f"```\n{o['pfad']}\n```\n")
    z(f"| Eigenschaft | Wert |")
    z(f"|:---|:---|")
    z(f"| Tiefe | {o['tiefe']} |")
    z(f"| Dateien | {o['anzahl_dateien']} |")
    z(f"| Unterordner | {o['anzahl_unterordner']} |")
    z(f"| Domäne | {o['vermutete_domaene']} |")
    if o.get('groesste_datei'):
        z(f"| Größte Datei | {o['groesste_datei']} |")
    z("")

    # Dateitypen
    if o.get('dateitypen'):
        z("### Dateitypen")
        z("")
        for ext, count in sorted(o['dateitypen'].items(), key=lambda x: -x[1]):
            z(f"- `{ext}`: {count}")
        z("")

    # Dateien im Ordner
    if o.get('dateien'):
        z("### Dateien")
        z("")
        z("| Name | Typ | Größe | Geändert |")
        z("|:---|:---|---:|:---|")
        for d in o['dateien'][:100]:
            groesse_mb = d['groesse_bytes'] / (1024*1024)
            groesse_str = f"{groesse_mb:.1f} MB" if groesse_mb >= 1 else f"{d['groesse_bytes']/1024:.0f} KB"
            flags = []
            if d.get('ist_generisch'):
                flags.append('🔴 generisch')
            if d.get('hat_versionierung'):
                flags.append('🟠 versioniert')
            if d.get('hat_problematische_zeichen'):
                flags.append('🔵 Sonderz.')
            flag_str = ' '.join(flags)
            name_display = f"{d['name']} {flag_str}" if flags else d['name']
            z(f"| {name_display} | `{d['erweiterung']}` | {groesse_str} | {d.get('geaendert', '?')} |")
        if len(o['dateien']) > 100:
            z(f"| *... +{len(o['dateien'])-100} weitere* | | | |")
        z("")

    # Probleme in diesem Ordner
    ordner_probs = [p for p in e.get('probleme_phase0', [])
                    if p.get('pfad', '').startswith(o['pfad'])]
    if ordner_probs:
        z(f"### Probleme in diesem Ordner: {len(ordner_probs)}")
        z("")
        for p in ordner_probs:
            z(_problem_zeile(p))
    else:
        z("### ✅ Keine Probleme in diesem Ordner")

    return "\n".join(zeilen)


def pfad_im_explorer_oeffnen(pfad: str) -> str:
    """Öffnet einen Pfad im Dateimanager des Betriebssystems."""
    if not pfad or not pfad.strip():
        return "Bitte einen Pfad eingeben."

    pfad = pfad.strip()
    p = Path(pfad)

    # Wenn es eine Datei ist, den Eltern-Ordner öffnen
    if p.is_file():
        ziel = p.parent
    elif p.is_dir():
        ziel = p
    else:
        return f"⚠️ Pfad existiert nicht: `{pfad}`"

    try:
        system = platform.system()
        if system == "Windows":
            # Explorer öffnen und Datei markieren (falls Datei)
            if p.is_file():
                subprocess.Popen(['explorer', '/select,', str(p)])
            else:
                os.startfile(str(ziel))
        elif system == "Darwin":  # macOS
            subprocess.Popen(['open', str(ziel)])
        else:  # Linux
            subprocess.Popen(['xdg-open', str(ziel)])
        return f"✅ Geöffnet: `{ziel}`"
    except Exception as ex:
        return f"⚠️ Fehler beim Öffnen: {ex}"


print("Detail- und Explorer-Funktionen geladen.")

## 5. Gradio-Interface starten (v2)

### Tabs:
- **Scan** — Pfad eingeben, Vorschau, Scan starten
- **Übersicht** — Statistik, Dateitypen, Severity-Zusammenfassung
- **🔴 S1 / 🟠 S2 / 🟡 S3 / 🔵 S4** — Probleme gefiltert, mit kopierbaren Pfaden
- **Duplikate** — Hash-Duplikate mit Pfaden
- **Ordner-Detail** — Einzelnen Ordner inspizieren
- **Explorer** — Pfad eingeben → im Explorer öffnen
- **Export** — JSON herunterladen (kompakt oder voll)

In [ ]:
# Globale Variable für das letzte Scan-Ergebnis
_letztes_ergebnis: dict = {}


def scan_ausfuehren(pfad: str, max_hash_mb: int):
    """Führt den Scan aus, gibt Übersicht + Severity-Berichte zurück."""
    global _letztes_ergebnis

    leer = "*Noch kein Scan durchgeführt.*"

    if not pfad or not pfad.strip():
        return ("⚠️ Bitte einen Pfad eingeben.",) + (leer,) * 6

    pfad = pfad.strip()
    p = Path(pfad)

    if not p.exists():
        return (f"⚠️ Pfad existiert nicht: `{p}`",) + (leer,) * 6
    if not p.is_dir():
        return (f"⚠️ Pfad ist kein Ordner: `{p}`",) + (leer,) * 6

    try:
        scanner = Phase0Scanner(pfad, max_hash_size_mb=max_hash_mb)
        ergebnis = scanner.scannen()
        _letztes_ergebnis = ergebnis

        uebersicht = bericht_uebersicht(ergebnis)
        s1 = bericht_severity(ergebnis, "S1")
        s2 = bericht_severity(ergebnis, "S2")
        s3 = bericht_severity(ergebnis, "S3")
        s4 = bericht_severity(ergebnis, "S4")
        dups = bericht_duplikate(ergebnis)

        n_probs = len(ergebnis['probleme_phase0'])
        status = f"✅ Scan abgeschlossen: {ergebnis['gesamtdateien']:,} Dateien, {ergebnis['gesamtordner']:,} Ordner, {n_probs} Probleme"

        return status, uebersicht, s1, s2, s3, s4, dups

    except Exception as ex:
        return (f"⚠️ Fehler: {ex}",) + (leer,) * 6


def ordner_detail_handler(pfad_suche: str) -> str:
    """Wrapper für Ordner-Detail, nutzt letztes Ergebnis."""
    if not _letztes_ergebnis:
        return "*Zuerst einen Scan durchführen.*"
    return ordner_detail(_letztes_ergebnis, pfad_suche)


def json_exportieren(kompakt: bool) -> Optional[str]:
    """Exportiert das Ergebnis als JSON-Datei."""
    if not _letztes_ergebnis:
        return None

    ergebnis = _letztes_ergebnis.copy()

    if kompakt:
        # Kompakt: ohne alle_dateien (spart ~90% Größe)
        ergebnis = {k: v for k, v in ergebnis.items() if k != 'alle_dateien'}
        # Auch dateien in ordnern entfernen
        for o in ergebnis.get('ordner', []):
            o.pop('dateien', None)
        suffix = "_kompakt"
    else:
        suffix = "_voll"

    pfad_name = Path(_letztes_ergebnis.get('eingabepfad', 'scan')).name or 'scan'
    json_pfad = os.path.join(tempfile.gettempdir(), f"{pfad_name}_phase0{suffix}.json")
    with open(json_pfad, "w", encoding="utf-8") as f:
        json.dump(ergebnis, f, ensure_ascii=False, indent=2)

    return json_pfad


# ─── Gradio-Interface ──────────────────────────────────────

with gr.Blocks(
    title="Phase 0 — Ist-Aufnahme v2",
    theme=gr.themes.Soft(),
) as app:

    gr.Markdown("# 🗂️ Phase 0: Ist-Aufnahme (v2)")
    gr.Markdown(
        "Scannt einen Ordner rekursiv und erstellt einen interaktiven Analysebericht. "
        "**Rein lesend** — es werden keine Dateien verändert.\n\n"
        "**Neu:** Pfade sind kopierbar · Severity-Filter · Explorer-Öffnung · Ordner-Detail"
    )

    # ─── Scan-Bereich ──────────────────────────────────────
    with gr.Row():
        with gr.Column(scale=3):
            pfad_eingabe = gr.Textbox(
                label="Ordnerpfad",
                placeholder="z.B. D:/Lehramt oder /Users/name/Documents",
                info="Pfad zum Ordner, der analysiert werden soll.",
            )
        with gr.Column(scale=1):
            hash_slider = gr.Slider(
                minimum=10, maximum=1000, value=200, step=10,
                label="Max. Hash-Größe (MB)",
                info="Dateien größer als dieser Wert werden nicht gehasht.",
            )

    with gr.Row():
        btn_vorschau = gr.Button("📂 Ordner-Vorschau", variant="secondary")
        btn_scan = gr.Button("🔍 Scan starten", variant="primary")

    vorschau_ausgabe = gr.Markdown(
        value="*Pfad eingeben und 'Ordner-Vorschau' klicken.*",
    )

    status_ausgabe = gr.Markdown(value="")

    # ─── Ergebnis-Tabs ─────────────────────────────────────
    with gr.Tabs():
        with gr.TabItem("📊 Übersicht"):
            tab_uebersicht = gr.Markdown(value="*Noch kein Scan durchgeführt.*")

        with gr.TabItem("🔴 S1 Kritisch"):
            tab_s1 = gr.Markdown(value="*Noch kein Scan durchgeführt.*")

        with gr.TabItem("🟠 S2 Hoch"):
            tab_s2 = gr.Markdown(value="*Noch kein Scan durchgeführt.*")

        with gr.TabItem("🟡 S3 Mittel"):
            tab_s3 = gr.Markdown(value="*Noch kein Scan durchgeführt.*")

        with gr.TabItem("🔵 S4 Niedrig"):
            tab_s4 = gr.Markdown(value="*Noch kein Scan durchgeführt.*")

        with gr.TabItem("📋 Duplikate"):
            tab_dups = gr.Markdown(value="*Noch kein Scan durchgeführt.*")

        with gr.TabItem("🔍 Ordner-Detail"):
            gr.Markdown("Pfad eingeben (oder aus Problembericht kopieren), um Ordner-Details zu sehen:")
            detail_eingabe = gr.Textbox(
                label="Ordnerpfad suchen",
                placeholder="Pfad aus Problembericht einfügen...",
            )
            btn_detail = gr.Button("🔍 Ordner anzeigen", variant="secondary")
            tab_detail = gr.Markdown(value="")

        with gr.TabItem("📂 Explorer öffnen"):
            gr.Markdown("Pfad eingeben → wird im Dateimanager geöffnet (Windows/macOS/Linux):")
            explorer_eingabe = gr.Textbox(
                label="Pfad zum Öffnen",
                placeholder="Pfad aus Problembericht einfügen...",
            )
            btn_explorer = gr.Button("📂 Im Explorer öffnen", variant="primary")
            explorer_status = gr.Markdown(value="")

        with gr.TabItem("💾 Export"):
            gr.Markdown(
                "### JSON-Export\n\n"
                "**Kompakt** = ohne Einzeldatei-Liste (\~90% kleiner, reicht für Phase 1)\n\n"
                "**Voll** = mit allen Dateidetails (für Archivierung / Debug)"
            )
            with gr.Row():
                btn_export_kompakt = gr.Button("💾 Kompakt-JSON", variant="primary")
                btn_export_voll = gr.Button("💾 Voll-JSON", variant="secondary")
            json_download = gr.File(label="Download")

    # ─── Events ────────────────────────────────────────────
    btn_vorschau.click(
        fn=ordner_vorschau,
        inputs=[pfad_eingabe],
        outputs=[vorschau_ausgabe],
    )

    btn_scan.click(
        fn=scan_ausfuehren,
        inputs=[pfad_eingabe, hash_slider],
        outputs=[status_ausgabe, tab_uebersicht, tab_s1, tab_s2, tab_s3, tab_s4, tab_dups],
    )

    btn_detail.click(
        fn=ordner_detail_handler,
        inputs=[detail_eingabe],
        outputs=[tab_detail],
    )

    btn_explorer.click(
        fn=pfad_im_explorer_oeffnen,
        inputs=[explorer_eingabe],
        outputs=[explorer_status],
    )

    btn_export_kompakt.click(
        fn=lambda: json_exportieren(kompakt=True),
        inputs=[],
        outputs=[json_download],
    )

    btn_export_voll.click(
        fn=lambda: json_exportieren(kompakt=False),
        inputs=[],
        outputs=[json_download],
    )

app.launch(inbrowser=True)